# 06B FORCE legacy RF FULL PRECISION FIXED

> Saved outputs are from an earlier local run. Portable path edits have **not** been re-executed. See the repository README for setup and scientific limitations.


In [1]:
import sys, subprocess, importlib.util
packages={"pandas":"pandas","numpy":"numpy","scipy":"scipy","sklearn":"scikit-learn","joblib":"joblib","pyarrow":"pyarrow"}
for module,package in packages.items():
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable,"-m","pip","install","-q",package])
print("Dependencies ready.")

Dependencies ready.


In [2]:
from pathlib import Path
from itertools import combinations
import gc, hashlib, json, platform, time, warnings
import joblib, numpy as np, pandas as pd, scipy, sklearn
from scipy.stats import spearmanr, wilcoxon
from sklearn.preprocessing import LabelEncoder
from IPython.display import display
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns",300)
pd.set_option("display.max_rows",300)
EPS=1e-12
EPSILON=EPS
ARGMAX_TIE_TOL=1e-12
MIN_WELLS_FOR_WILCOXON=5
RANDOM_STATE=142

In [3]:
RUN_MODE="full"  # "pilot" or "full"
ALL_FOLDS=[1,2,3,4,5]
ACTIVE_FOLDS=[1] if RUN_MODE=="pilot" else ALL_FOLDS
DOSES=[0.25,0.50,0.75,1.00]
RAW8_FEATURES=["GR","NPHI","RHOB","DTC","RDEP","CALI","PEF","SP"]
SAVE_ROW_LEVEL_RESULTS=(RUN_MODE=="pilot")
PREDICTION_BATCH_SIZE=100_000
BOOTSTRAP_REPLICATES=2000 if RUN_MODE=="pilot" else 1000
SKIP_COMPLETED_CONDITIONS=True

# Portable paths: override them with environment variables if upstream artifacts
# live outside this repository. See README -> Reproduction & paths.
import os

def _find_stratashift_root():
    start = Path(os.environ.get("STRATASHIFT_REPO_ROOT", Path.cwd())).expanduser().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "notebooks" / "interventions").is_dir() and (candidate / "README.md").is_file():
            return candidate
    raise FileNotFoundError(
        "StrataShift root not found. Launch Jupyter inside the repository "
        "or set STRATASHIFT_REPO_ROOT."
    )

REPO_ROOT = _find_stratashift_root()
TRAIN_PATH = Path(os.environ.get(
    "STRATASHIFT_TRAIN_CSV", str(REPO_ROOT / "data" / "raw" / "train.csv")
)).expanduser().resolve()
PROJECT = Path(os.environ.get(
    "STRATASHIFT_ARTIFACT_ROOT", str(REPO_ROOT / "artifacts")
)).expanduser().resolve()
FOLD_PATH=PROJECT/"01_clean_splits_support"/"10_grouped_5fold_well_assignment.csv"
D02B=PROJECT/"02B_legacy_RF_grouped_well_CV"/"full"
ARTIFACT_DIR=D02B/"artifacts"
OOF02B=D02B/"oof_predictions"
CONFIG02B=D02B/"12_run_configuration.json"
D03=PROJECT/"03_nested_grouped_temperature_calibration"/"full"
TEMP_PATH=D03/"10_nested_temperature_registry.json"
GATE03=D03/"09_nested_calibration_gate.json"
CONFIG03=D03/"11_run_configuration.json"
CAL_OOF03=D03/"calibrated_outer_oof"
D04C=PROJECT/"04C_shared_class_conditional_transport_bank"/"full"
BANK_DIR=D04C/"transport_bank"
GATE04C=D04C/"07_transport_bank_gate.json"
D06A=PROJECT/"06A_raw_RF_class_conditional_transport_interventions"/RUN_MODE
SELECTED_PATH=D06A/"01_selected_intervention_conditions.csv"
GATE06A=D06A/"11_stage_gate.json"
OUT=PROJECT/"06B_legacy_RF_class_conditional_transport_interventions"/RUN_MODE
COND_DIR=OUT/"condition_outputs"
WELL_DIR=OUT/"per_well"
ROW_DIR=OUT/"row_level_results"
PLOT_DIR=OUT/"plots"
for d in [OUT,COND_DIR,WELL_DIR,ROW_DIR,PLOT_DIR]: d.mkdir(parents=True,exist_ok=True)
required=[TRAIN_PATH,FOLD_PATH,ARTIFACT_DIR,OOF02B,CONFIG02B,TEMP_PATH,GATE03,CONFIG03,CAL_OOF03,BANK_DIR,GATE04C,SELECTED_PATH,GATE06A]
missing=[str(p) for p in required if not p.exists()]
if missing: raise FileNotFoundError("Missing inputs:\n"+"\n".join(missing))
print("RUN_MODE:",RUN_MODE,"folds:",ACTIVE_FOLDS,"output:",OUT)

RUN_MODE: full folds: [1, 2, 3, 4, 5] output: <ARTIFACT_ROOT>/06B_legacy_RF_class_conditional_transport_interventions/full


In [4]:
TARGET_COLUMN="FORCE_2020_LITHOFACIES_LITHOLOGY"
CONFIDENCE_COLUMN="FORCE_2020_LITHOFACIES_CONFIDENCE"
CLASSES=np.array([30000,65000,65030,70000,70032,74000,80000,86000,88000,90000,93000,99000],dtype=np.int64)
NAMES={30000:"Sandstone",65000:"Shale",65030:"Sandstone/Shale",70000:"Limestone",70032:"Chalk",74000:"Dolomite",80000:"Marl",86000:"Anhydrite",88000:"Halite",90000:"Coal",93000:"Basement",99000:"Tuff"}
CLASS_TO_INDEX={int(c):i for i,c in enumerate(CLASSES)}
CLASS_INDICES=list(range(len(CLASSES)))
CLASS_CODES=CLASSES.tolist()
LITHOLOGY_NAMES=NAMES
CLASS_INDEX_TO_NAME={i:NAMES[int(c)] for i,c in enumerate(CLASSES)}
label_encoder=LabelEncoder().fit(CLASSES)

BASE_CAUSAL_FEATURES = [
    "GR",
    "NPHI",
    "RHOB",
    "PEF",
    "DTC",
    "SP",
]

TOP_LEGACY_FEATURES=[
 "GR_NPHI_interaction","GR_DTC_interaction","Y_LOC","GR_RHOB_RATIO","NPHI_DTC_interaction","X_LOC",
 "RHOB_DTC_interaction","CALI","Z_LOC","GR_RHOB_interaction","DEPTH_NORMALIZED","DEPTH_MD","RDEP",
 "RHOB_log","RMED","RHOB","X_LOC"]
EXTRA_WEAK_CLASS_FEATURES=["GR_flatness","DTC_grad","DTC_RHOB_DIFF","DTC_RHOB_RATIO2","RDEP_GR_RATIO","RHOB_NPHI_DIFF2","Acoustic_Impedance","Sand_indicator","Shale_indicator"]
LEGACY_FEATURE_COLUMNS=TOP_LEGACY_FEATURES+EXTRA_WEAK_CLASS_FEATURES
assert len(LEGACY_FEATURE_COLUMNS)==26 and len(set(LEGACY_FEATURE_COLUMNS))==25 and LEGACY_FEATURE_COLUMNS.count("X_LOC")==2
FEATURE_POSITIONS={}
for j,name in enumerate(LEGACY_FEATURE_COLUMNS): FEATURE_POSITIONS.setdefault(name,[]).append(j)

DESCENDANTS={
 "GR":["GR_NPHI_interaction","GR_DTC_interaction","GR_RHOB_RATIO","GR_RHOB_interaction","GR_flatness","RDEP_GR_RATIO","Shale_indicator"],
 "NPHI":["GR_NPHI_interaction","NPHI_DTC_interaction","RHOB_NPHI_DIFF2","Shale_indicator"],
 "RHOB":["GR_RHOB_RATIO","RHOB_DTC_interaction","GR_RHOB_interaction","RHOB_log","RHOB","DTC_RHOB_DIFF","DTC_RHOB_RATIO2","RHOB_NPHI_DIFF2","Acoustic_Impedance","Sand_indicator","Shale_indicator"],
 "DTC":["GR_DTC_interaction","NPHI_DTC_interaction","RHOB_DTC_interaction","DTC_grad","DTC_RHOB_DIFF","DTC_RHOB_RATIO2","Acoustic_Impedance","Sand_indicator"],
 "RDEP":["RDEP","RDEP_GR_RATIO"],"CALI":["CALI"],"PEF":[],"SP":[]}
STRUCTURAL_ZERO=["PEF","SP"]
descendant_map=pd.DataFrame([{"raw_feature":f,"model_descendant":d,"structural_zero":len(DESCENDANTS[f])==0} for f in RAW8_FEATURES for d in (DESCENDANTS[f] or [None])])
display(descendant_map)

,raw_feature,model_descendant,structural_zero
0,GR,GR_NPHI_interaction,False
1,GR,GR_DTC_interaction,False
2,GR,GR_RHOB_RATIO,False
3,GR,GR_RHOB_interaction,False
4,GR,GR_flatness,False
5,GR,RDEP_GR_RATIO,False
6,GR,Shale_indicator,False
7,NPHI,GR_NPHI_interaction,False
8,NPHI,NPHI_DTC_interaction,False
9,NPHI,RHOB_NPHI_DIFF2,False


In [5]:
def sha256_file(path,chunk=1024*1024):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        while True:
            b=f.read(chunk)
            if not b: break
            h.update(b)
    return h.hexdigest()

def dose_col(alpha): return f"value_alpha_{float(alpha):.2f}".replace(".","_")
def stable_seed(*parts): return int(hashlib.sha256("||".join(map(str,parts)).encode()).hexdigest()[:8],16)

def bootstrap_ci(values,reps,seed):
    v=np.asarray(values,float); v=v[np.isfinite(v)]
    if len(v)==0:return np.nan,np.nan,np.nan
    if len(v)==1:return float(v[0]),float(v[0]),float(v[0])
    rng=np.random.default_rng(seed)
    means=v[rng.integers(0,len(v),size=(reps,len(v)))].mean(1)
    return float(v.mean()),float(np.quantile(means,.025)),float(np.quantile(means,.975))

def safe_wilcoxon(values):
    v=np.asarray(values,float); v=v[np.isfinite(v)]
    if len(v)<MIN_WELLS_FOR_WILCOXON:return np.nan
    if np.allclose(v,0,rtol=0,atol=1e-14):return 1.0
    try:return float(wilcoxon(v,zero_method="wilcox",alternative="two-sided",method="auto").pvalue)
    except ValueError:return 1.0

def bh(p):
    p=np.asarray(p,float); q=np.full(len(p),np.nan); idx=np.where(np.isfinite(p))[0]
    if len(idx)==0:return q
    order=np.argsort(p[idx]); ranked=p[idx][order]; m=len(ranked)
    adj=np.minimum.accumulate((ranked*m/np.arange(1,m+1))[::-1])[::-1]
    q[idx[order]]=np.clip(adj,0,1)
    return q

def json_safe(x):
    if isinstance(x,dict):return {str(k):json_safe(v) for k,v in x.items()}
    if isinstance(x,(list,tuple)):return [json_safe(v) for v in x]
    if isinstance(x,np.ndarray):return x.tolist()
    if isinstance(x,np.generic):return x.item()
    if isinstance(x,Path):return str(x)
    if isinstance(x,float) and not np.isfinite(x):return None
    return x

def align_probabilities(local,local_classes):
    out=np.zeros((len(local),len(CLASSES)),dtype=np.float64)
    for j,c in enumerate(local_classes): out[:,CLASS_TO_INDEX[int(c)]]=local[:,j]
    return out/out.sum(1,keepdims=True)

def predict_batches(model,X,batch=PREDICTION_BATCH_SIZE):
    parts=[]
    for start in range(0,len(X),batch):
        local=model.predict_proba(X[start:start+batch])
        parts.append(align_probabilities(local,model.classes_))
    return np.vstack(parts)

def apply_temperature(P,T):
    Z=np.log(np.clip(np.asarray(P,float),EPS,1))/float(T)
    Z-=Z.max(1,keepdims=True)
    E=np.exp(Z)
    return E/E.sum(1,keepdims=True)

def temperature_argmax_audit(P_raw,P_cal,tolerance=ARGMAX_TIE_TOL):
    P_raw=np.asarray(P_raw,dtype=np.float64)
    P_cal=np.asarray(P_cal,dtype=np.float64)

    raw_index=np.argmax(P_raw,axis=1)
    calibrated_index=np.argmax(P_cal,axis=1)
    mismatch=raw_index!=calibrated_index

    top_two=np.partition(P_raw,-2,axis=1)[:,-2:]
    raw_top2_margin=top_two[:,1]-top_two[:,0]

    non_tie_mismatch=(
        mismatch
        & (raw_top2_margin>tolerance)
    )

    return {
      "probabilities_valid":bool(
          np.isfinite(P_raw).all()
          and np.isfinite(P_cal).all()
          and np.allclose(P_raw.sum(1),1,rtol=1e-6,atol=1e-6)
          and np.allclose(P_cal.sum(1),1,rtol=1e-6,atol=1e-6)
      ),
      "argmax_mismatch_rows":int(mismatch.sum()),
      "argmax_mismatch_rate":float(mismatch.mean()),
      "non_tie_argmax_mismatch_rows":int(non_tie_mismatch.sum()),
      "all_argmax_mismatches_are_near_ties":bool(
          not non_tie_mismatch.any()
      ),
      "maximum_raw_top2_margin_among_mismatches":float(
          raw_top2_margin[mismatch].max()
      ) if mismatch.any() else 0.0,
      "tolerance":float(tolerance),
    }

def bootstrap_ci(values,reps,seed):
    v=np.asarray(values,float); v=v[np.isfinite(v)]
    if len(v)==0: return np.nan,np.nan,np.nan
    if len(v)==1: return float(v[0]),float(v[0]),float(v[0])
    rng=np.random.default_rng(seed)
    means=v[rng.integers(0,len(v),size=(reps,len(v)))].mean(1)
    return float(v.mean()),float(np.quantile(means,.025)),float(np.quantile(means,.975))

def safe_wilcoxon(values):
    v=np.asarray(values,float); v=v[np.isfinite(v)]
    if len(v)<MIN_WELLS_FOR_WILCOXON: return np.nan
    if np.allclose(v,0,rtol=0,atol=1e-14): return 1.0
    try: return float(wilcoxon(v,zero_method="wilcox",alternative="two-sided",method="auto").pvalue)
    except ValueError: return 1.0

def bh(p):
    p=np.asarray(p,float); q=np.full(len(p),np.nan); idx=np.where(np.isfinite(p))[0]
    if len(idx)==0:return q
    order=np.argsort(p[idx]); ranked=p[idx][order]; m=len(ranked)
    adj=np.minimum.accumulate((ranked*m/np.arange(1,m+1))[::-1])[::-1]
    q[idx[order]]=np.clip(adj,0,1)
    return q

In [6]:
with open(CONFIG02B,encoding="utf-8") as f:cfg02=json.load(f)
with open(GATE03,encoding="utf-8") as f:g03=json.load(f)
with open(CONFIG03,encoding="utf-8") as f:cfg03=json.load(f)
with open(TEMP_PATH,encoding="utf-8") as f:temperature_registry=json.load(f)
with open(GATE04C,encoding="utf-8") as f:g04=json.load(f)
with open(GATE06A,encoding="utf-8") as f:g06=json.load(f)
upstream={
 "02B_legacy_model_unchanged":bool(cfg02.get("legacy_model_unchanged",False)),
 "02B_feature_count_26":cfg02.get("feature_column_count")==26,
 "03_nested_calibration_passed":bool(g03.get("overall_nested_calibration_passed",False)),
 "03_refined_labels_unused":cfg03.get("legacy_refined_labels_used_for_calibration") is False,
 "04C_transport_bank_passed":bool(g04.get("overall_transport_bank_passed",False)),
 "06A_stage_passed":bool(g06.get("overall_raw_RF_transport_intervention_stage_passed",False)),
 "06A_mode_matches":g06.get("run_mode")==RUN_MODE,
 "temperatures_complete":set(temperature_registry)==set(map(str,ALL_FOLDS)),
 "official_test_unused":True}
print(json.dumps(upstream,indent=2))
if not all(upstream.values()):raise RuntimeError("Upstream gate failed.")

{
  "02B_legacy_model_unchanged": true,
  "02B_feature_count_26": true,
  "03_nested_calibration_passed": true,
  "03_refined_labels_unused": true,
  "04C_transport_bank_passed": true,
  "06A_stage_passed": true,
  "06A_mode_matches": true,
  "temperatures_complete": true,
  "official_test_unused": true
}


In [7]:
def detect_csv(path):
    for enc in ["utf-8","latin1","iso-8859-1","cp1252"]:
        try:
            with open(path,encoding=enc) as f: lines=[f.readline() for _ in range(5)]
            sep=";" if any(";" in x for x in lines) else ("\t" if any("\t" in x for x in lines) else ",")
            return sep,enc
        except UnicodeDecodeError:pass
    return ",","utf-8"

sep,enc=detect_csv(TRAIN_PATH)
train=pd.read_csv(TRAIN_PATH,sep=sep,encoding=enc,low_memory=False)
train["ORIGINAL_ROW_ID"]=np.arange(len(train),dtype=np.int64)
folds=pd.read_csv(FOLD_PATH)
wc=[c for c in folds if c.upper()=="WELL"][0]
fc=[c for c in folds if "FOLD" in c.upper() and "SCORE" not in c.upper()][0]
fold_map=dict(zip(folds[wc],pd.to_numeric(folds[fc],errors="raise").astype(int)))
train["GROUP_FOLD"]=train.WELL.map(fold_map)
if train.GROUP_FOLD.isna().any():raise RuntimeError("Missing fold assignments.")
train["GROUP_FOLD"]=train.GROUP_FOLD.astype(int)
train[TARGET_COLUMN]=pd.to_numeric(train[TARGET_COLUMN],errors="raise").astype(int)

selected=pd.read_csv(SELECTED_PATH)
selected["outer_fold"]=pd.to_numeric(selected.outer_fold,errors="raise").astype(int)
selected=selected[selected.outer_fold.isin(ACTIVE_FOLDS)].copy().reset_index(drop=True)
selected["resolved_bank_path"]=selected.condition_id.map(lambda x:str(BANK_DIR/f"{x}.parquet"))
if not selected.resolved_bank_path.map(lambda x:Path(x).exists()).all():raise FileNotFoundError("Missing intervention bank.")
expected_conditions=8 if RUN_MODE=="pilot" else 319
if len(selected)!=expected_conditions:raise RuntimeError(f"Expected {expected_conditions} conditions, found {len(selected)}")
print("train",train.shape,"conditions",len(selected))

train (1170511, 31) conditions 319


# 5. Exact Legacy preprocessing

In [8]:
def apply_train_imputers_to_validation(
    dataframe,
    imputers,
):
    df = dataframe.copy()

    if CONFIDENCE_COLUMN in df.columns:
        df = df.drop(
            columns=[CONFIDENCE_COLUMN]
        )

    numeric_columns = (
        df.select_dtypes(
            include=[np.number]
        )
        .columns
        .tolist()
    )

    for excluded in [
        TARGET_COLUMN,
        "LITHOLOGY_ENCODED",
        "GROUP_FOLD",
    ]:
        if excluded in numeric_columns:
            numeric_columns.remove(excluded)

    for column in numeric_columns:
        if (
            column in imputers
            and df[column].isna().any()
        ):
            df[column] = (
                imputers[column]
                .transform(
                    df[[column]]
                )
                .ravel()
            )

    df[numeric_columns] = (
        df[numeric_columns]
        .fillna(0)
        .replace(
            [np.inf, -np.inf],
            0,
        )
    )

    return df


def add_legacy_preprocess_features(
    dataframe,
):
    df = dataframe.copy()
    epsilon = 1e-10

    if {"NPHI", "RHOB"}.issubset(df.columns):
        df["NPHI_RHOB_RATIO"] = (
            df["NPHI"]
            / (
                df["RHOB"].abs()
                + epsilon
            )
        )

    if {"GR", "RHOB"}.issubset(df.columns):
        df["GR_RHOB_RATIO"] = (
            df["GR"]
            / (
                df["RHOB"].abs()
                + epsilon
            )
        )

    if {"DTC", "RHOB"}.issubset(df.columns):
        df["DTC_RHOB_RATIO"] = (
            df["DTC"]
            / (
                df["RHOB"].abs()
                + epsilon
            )
        )

    if "DEPTH_MD" in df.columns:
        depth_min = df["DEPTH_MD"].min()
        depth_max = df["DEPTH_MD"].max()
        depth_range = depth_max - depth_min

        if depth_range > 0:
            df["DEPTH_NORMALIZED"] = (
                (
                    df["DEPTH_MD"]
                    - depth_min
                )
                / depth_range
            )
        else:
            df["DEPTH_NORMALIZED"] = 0.0

    numeric_columns = (
        df.select_dtypes(
            include=[np.number]
        )
        .columns
    )

    df[numeric_columns] = (
        df[numeric_columns]
        .fillna(0)
        .replace(
            [np.inf, -np.inf],
            0,
        )
    )

    return df


def encode_target(
    dataframe,
):
    df = dataframe.copy()

    numeric_target = (
        pd.to_numeric(
            df[TARGET_COLUMN],
            errors="raise",
        )
        .astype(int)
    )

    unknown_codes = sorted(
        set(numeric_target)
        - set(CLASS_CODES)
    )

    if unknown_codes:
        raise ValueError(
            f"Unknown lithology codes: {unknown_codes}"
        )

    df[TARGET_COLUMN] = numeric_target

    df["LITHOLOGY_ENCODED"] = (
        label_encoder.transform(
            numeric_target
        )
    )

    return df

def compute_legacy_causal_features(
    dataframe,
):
    df = dataframe.copy()

    available = [
        feature
        for feature in BASE_CAUSAL_FEATURES
        if feature in df.columns
    ]

    for feature_1, feature_2 in combinations(
        available,
        2,
    ):
        df[
            f"{feature_1}_{feature_2}_interaction"
        ] = (
            df[feature_1]
            * df[feature_2]
        )

    for feature in available:
        df[
            f"{feature}_squared"
        ] = (
            df[feature] ** 2
        )

        df[
            f"{feature}_log"
        ] = np.log(
            np.abs(
                df[feature]
            )
            + 1
        )

    return df


def add_legacy_weak_class_features(
    dataframe,
):
    df = dataframe.copy()
    epsilon = 1e-6

    def has_columns(columns):
        return all(
            column in df.columns
            for column in columns
        )

    if has_columns(["GR"]):
        if "WELL" in df.columns:
            df["GR_grad"] = (
                df.groupby(
                    "WELL"
                )["GR"]
                .diff()
                .fillna(0)
            )

            df["GR_flatness"] = (
                df.groupby(
                    "WELL"
                )["GR"]
                .rolling(
                    window=5,
                    min_periods=1,
                )
                .std()
                .reset_index(
                    level=0,
                    drop=True,
                )
            )
        else:
            df["GR_grad"] = (
                df["GR"]
                .diff()
                .fillna(0)
            )

            df["GR_flatness"] = (
                df["GR"]
                .rolling(
                    5,
                    min_periods=1,
                )
                .std()
            )

    if has_columns(["DTC"]):
        if "WELL" in df.columns:
            df["DTC_grad"] = (
                df.groupby(
                    "WELL"
                )["DTC"]
                .diff()
                .fillna(0)
            )
        else:
            df["DTC_grad"] = (
                df["DTC"]
                .diff()
                .fillna(0)
            )

    if has_columns(["RDEP"]):
        if "WELL" in df.columns:
            df["RDEP_grad"] = (
                df.groupby(
                    "WELL"
                )["RDEP"]
                .diff()
                .fillna(0)
            )
        else:
            df["RDEP_grad"] = (
                df["RDEP"]
                .diff()
                .fillna(0)
            )

    if has_columns(["DTC", "RHOB"]):
        df["DTC_RHOB_DIFF"] = (
            df["DTC"]
            - df["RHOB"]
        )

        df["DTC_RHOB_RATIO2"] = (
            df["DTC"]
            / (
                df["RHOB"]
                + epsilon
            )
        )

    if has_columns(["RDEP", "GR"]):
        df["RDEP_GR_RATIO"] = (
            df["RDEP"]
            / (
                df["GR"]
                + epsilon
            )
        )

    if has_columns(["RHOB", "NPHI"]):
        df["RHOB_NPHI_DIFF2"] = (
            df["RHOB"]
            - df["NPHI"]
        )

    if has_columns(["GR"]):
        gr_min = df["GR"].min()
        gr_max = df["GR"].max()

        df["VSH_linear"] = (
            (
                df["GR"]
                - gr_min
            )
            / (
                gr_max
                - gr_min
                + epsilon
            )
        )

        df["VSH_Larionov"] = (
            0.083
            * (
                2 ** (
                    3.7
                    * df["VSH_linear"]
                )
                - 1
            )
        )

        df["VSH_Stieber"] = (
            df["VSH_linear"]
            / (
                3
                - 2
                * df["VSH_linear"]
                + epsilon
            )
        )

    if has_columns(["DTC"]):
        df["Vp"] = (
            1e6
            / (
                df["DTC"]
                + epsilon
            )
        )

    if has_columns(["RHOB", "Vp"]):
        df["Acoustic_Impedance"] = (
            df["RHOB"]
            * df["Vp"]
        )

    if has_columns(["NPHI", "RHOB"]):
        df["Poisson_proxy"] = (
            df["NPHI"]
            / (
                df["RHOB"]
                + epsilon
            )
        )

    if "NPHI" in df.columns:
        df["Brittleness"] = (
            1
            - df["NPHI"]
        )

    if has_columns(["Vp", "RHOB"]):
        df["Sand_indicator"] = (
            df["Vp"]
            / (
                df["RHOB"]
                + epsilon
            )
        )

    if has_columns([
        "RHOB",
        "NPHI",
        "VSH_Larionov",
    ]):
        df["Shale_indicator"] = (
            df["RHOB"]
            * df["NPHI"]
            * df["VSH_Larionov"]
        )

    numeric_columns = (
        df.select_dtypes(
            include=[np.number]
        )
        .columns
    )

    df[numeric_columns] = (
        df[numeric_columns]
        .fillna(0)
        .replace(
            [np.inf, -np.inf],
            0,
        )
    )

    return df

In [9]:
def align_probabilities(
    raw_probabilities,
    trained_classes,
):
    aligned = np.zeros(
        (
            raw_probabilities.shape[0],
            len(CLASS_INDICES),
        ),
        dtype=np.float32,
    )

    for source_column, class_index in enumerate(
        trained_classes
    ):
        aligned[
            :,
            int(class_index),
        ] = raw_probabilities[
            :,
            source_column,
        ]

    row_sums = aligned.sum(
        axis=1,
        keepdims=True,
    )

    aligned = np.divide(
        aligned,
        row_sums,
        out=np.zeros_like(aligned),
        where=row_sums > 0,
    )

    return aligned


def probabilities_to_log_scores(
    probabilities,
):
    clipped = np.clip(
        probabilities,
        EPSILON,
        1.0,
    )

    return np.log(
        clipped
    )


def softmax(
    scores,
):
    shifted = (
        scores
        - np.max(
            scores,
            axis=1,
            keepdims=True,
        )
    )

    exponentials = np.exp(
        shifted
    )

    return (
        exponentials
        / exponentials.sum(
            axis=1,
            keepdims=True,
        )
    )


def apply_temperature(
    probabilities,
    temperature,
):
    if (
        not np.isfinite(
            temperature
        )
        or temperature <= 0
    ):
        raise ValueError(
            f"Invalid temperature: {temperature}"
        )

    log_scores = (
        probabilities_to_log_scores(
            probabilities
        )
    )

    return softmax(
        log_scores
        / temperature
    )


def build_legacy_validation_matrix(
    raw_validation_dataframe,
    train_imputers,
):
    validation_imputed = (
        apply_train_imputers_to_validation(
            raw_validation_dataframe,
            imputers=train_imputers,
        )
    )

    # رفتار Legacy: Validation Fold از Min/Max خودش استفاده می‌کند.
    validation_preprocessed = (
        add_legacy_preprocess_features(
            validation_imputed
        )
    )

    validation_preprocessed = (
        encode_target(
            validation_preprocessed
        )
    )

    validation_causal = (
        compute_legacy_causal_features(
            validation_preprocessed
        )
    )

    # رفتار Legacy: WELL هنگام Weak Feature Engineering موجود است.
    validation_with_weak = (
        add_legacy_weak_class_features(
            validation_causal
        )
    )

    missing_features = [
        feature
        for feature in set(
            LEGACY_FEATURE_COLUMNS
        )
        if feature
        not in validation_with_weak.columns
    ]

    if missing_features:
        raise RuntimeError(
            "Missing Validation features: "
            f"{missing_features}"
        )

    X_validation = (
        validation_with_weak[
            LEGACY_FEATURE_COLUMNS
        ]
        .copy()
    )

    y_validation = (
        validation_with_weak[
            "LITHOLOGY_ENCODED"
        ]
        .astype(int)
        .to_numpy()
    )

    metadata_columns = [
        column
        for column in [
            "ORIGINAL_ROW_ID",
            "WELL",
            "DEPTH_MD",
            TARGET_COLUMN,
            "LITHOLOGY_ENCODED",
        ]
        if column
        in validation_with_weak.columns
    ]

    metadata = (
        validation_with_weak[
            metadata_columns
        ]
        .copy()
        .reset_index(drop=True)
    )

    return {
        "X": X_validation,
        "y": y_validation,
        "metadata": metadata,
    }


def predict_aligned_in_batches(
    model,
    scaled_features,
    batch_size=100_000,
):
    n_rows = scaled_features.shape[0]
    output = np.zeros(
        (
            n_rows,
            len(CLASS_INDICES),
        ),
        dtype=np.float32,
    )

    for start in range(
        0,
        n_rows,
        batch_size,
    ):
        stop = min(
            start + batch_size,
            n_rows,
        )

        partial = model.predict_proba(
            scaled_features[
                start:stop
            ]
        )

        output[
            start:stop
        ] = align_probabilities(
            partial,
            model.classes_,
        ).astype(np.float32)

    return output


def true_class_probability(
    probabilities,
    y_true,
):
    return probabilities[
        np.arange(len(y_true)),
        y_true,
    ]


def per_row_log_loss(
    probabilities,
    y_true,
):
    true_probability = (
        true_class_probability(
            probabilities,
            y_true,
        )
    )

    return -np.log(
        np.clip(
            true_probability,
            EPSILON,
            1.0,
        )
    )

In [10]:
required_legacy_globals = {
    "BASE_CAUSAL_FEATURES": BASE_CAUSAL_FEATURES,
    "EPSILON": EPSILON,
    "LEGACY_FEATURE_COLUMNS": LEGACY_FEATURE_COLUMNS,
}

assert BASE_CAUSAL_FEATURES == [
    "GR",
    "NPHI",
    "RHOB",
    "PEF",
    "DTC",
    "SP",
]

assert EPSILON == EPS
assert len(LEGACY_FEATURE_COLUMNS) == 26
assert len(set(LEGACY_FEATURE_COLUMNS)) == 25
assert LEGACY_FEATURE_COLUMNS.count("X_LOC") == 2

_test_feature_frame = pd.DataFrame({
    "GR": [100.0, 110.0],
    "NPHI": [0.20, 0.25],
    "RHOB": [2.30, 2.40],
    "PEF": [3.0, 3.2],
    "DTC": [80.0, 85.0],
    "SP": [-20.0, -18.0],
})

_test_engineered = compute_legacy_causal_features(
    _test_feature_frame
)

_expected_test_columns = {
    "GR_NPHI_interaction",
    "GR_DTC_interaction",
    "NPHI_DTC_interaction",
    "RHOB_DTC_interaction",
    "RHOB_log",
}

_missing_test_columns = (
    _expected_test_columns
    - set(_test_engineered.columns)
)

if _missing_test_columns:
    raise RuntimeError(
        "Legacy causal feature preflight failed. Missing: "
        f"{sorted(_missing_test_columns)}"
    )

del (
    required_legacy_globals,
    _test_feature_frame,
    _test_engineered,
    _expected_test_columns,
    _missing_test_columns,
)

print(
    "Legacy feature-engineering preflight passed: "
    "BASE_CAUSAL_FEATURES and EPSILON are defined."
)

Legacy feature-engineering preflight passed: BASE_CAUSAL_FEATURES and EPSILON are defined.


# 6. Independent-row descendant recomputation

In [11]:
def extrema_excluding_each(values):
    v=np.asarray(values,float); u,c=np.unique(v,return_counts=True)
    if len(u)==1:return np.full(len(v),u[0]),np.full(len(v),u[0])
    lo=np.full(len(v),u[0]); hi=np.full(len(v),u[-1])
    if c[0]==1:lo[v==u[0]]=u[1]
    if c[-1]==1:hi[v==u[-1]]=u[-2]
    return lo,hi

def build_context(imputed):
    ctx={f:pd.to_numeric(imputed[f],errors="raise").to_numpy(float) for f in RAW8_FEATURES}
    wells=imputed.WELL.astype(str)
    gr=pd.Series(ctx["GR"],index=imputed.index).groupby(wells,sort=False)
    dtc=pd.Series(ctx["DTC"],index=imputed.index).groupby(wells,sort=False)
    for k in range(1,5):ctx[f"GR_PREV_{k}"]=gr.shift(k).to_numpy(float)
    ctx["DTC_PREV_1"]=dtc.shift(1).to_numpy(float)
    ctx["GR_MIN_EXCL"],ctx["GR_MAX_EXCL"]=extrema_excluding_each(ctx["GR"])
    return ctx

def rolling_gr_std(ctx,pos,cf):
    vals=np.column_stack([ctx["GR_PREV_4"][pos],ctx["GR_PREV_3"][pos],ctx["GR_PREV_2"][pos],ctx["GR_PREV_1"][pos],cf])
    finite=np.isfinite(vals); n=finite.sum(1); means=np.divide(np.nansum(vals,1),n,out=np.zeros(len(pos)),where=n>0)
    ss=np.sum(np.where(finite,vals-means[:,None],0.0)**2,1)
    var=np.divide(ss,n-1,out=np.zeros(len(pos)),where=n>1)
    return np.sqrt(np.maximum(var,0))

def set_feature(X,name,values):
    for j in FEATURE_POSITIONS[name]:X[:,j]=values

def recompute_descendants(X0,feature,cf,pos,ctx):
    X=np.asarray(X0,float).copy(); cf=np.asarray(cf,float); pos=np.asarray(pos,np.int64)
    gr=ctx["GR"][pos].copy(); nphi=ctx["NPHI"][pos].copy(); rhob=ctx["RHOB"][pos].copy()
    dtc=ctx["DTC"][pos].copy(); rdep=ctx["RDEP"][pos].copy(); cali=ctx["CALI"][pos].copy()
    if feature=="GR":gr=cf
    elif feature=="NPHI":nphi=cf
    elif feature=="RHOB":rhob=cf
    elif feature=="DTC":dtc=cf
    elif feature=="RDEP":rdep=cf
    elif feature=="CALI":cali=cf
    elif feature in STRUCTURAL_ZERO:return X
    else:raise KeyError(feature)
    ep=1e-10; ew=1e-6
    if feature in ["GR","NPHI"]:set_feature(X,"GR_NPHI_interaction",gr*nphi)
    if feature in ["GR","DTC"]:set_feature(X,"GR_DTC_interaction",gr*dtc)
    if feature in ["GR","RHOB"]:
        set_feature(X,"GR_RHOB_RATIO",gr/(np.abs(rhob)+ep)); set_feature(X,"GR_RHOB_interaction",gr*rhob)
    if feature in ["NPHI","DTC"]:set_feature(X,"NPHI_DTC_interaction",nphi*dtc)
    if feature in ["RHOB","DTC"]:
        set_feature(X,"RHOB_DTC_interaction",rhob*dtc); set_feature(X,"DTC_RHOB_DIFF",dtc-rhob)
        set_feature(X,"DTC_RHOB_RATIO2",dtc/(rhob+ew))
        vp=1e6/(dtc+ew); set_feature(X,"Acoustic_Impedance",rhob*vp); set_feature(X,"Sand_indicator",vp/(rhob+ew))
    if feature=="CALI":set_feature(X,"CALI",cali)
    if feature=="RDEP":set_feature(X,"RDEP",rdep)
    if feature=="RHOB":set_feature(X,"RHOB_log",np.log(np.abs(rhob)+1)); set_feature(X,"RHOB",rhob)
    if feature=="GR":set_feature(X,"GR_flatness",rolling_gr_std(ctx,pos,gr))
    if feature=="DTC":
        prev=ctx["DTC_PREV_1"][pos]; set_feature(X,"DTC_grad",np.where(np.isfinite(prev),dtc-prev,0.0))
    if feature in ["GR","RDEP"]:set_feature(X,"RDEP_GR_RATIO",rdep/(gr+ew))
    if feature in ["RHOB","NPHI"]:set_feature(X,"RHOB_NPHI_DIFF2",rhob-nphi)
    if feature in ["GR","RHOB","NPHI"]:
        if feature=="GR":
            mn=np.minimum(gr,ctx["GR_MIN_EXCL"][pos]); mx=np.maximum(gr,ctx["GR_MAX_EXCL"][pos])
            vsh=(gr-mn)/(mx-mn+ew)
        else:
            allgr=ctx["GR"]; vsh=(gr-allgr.min())/(allgr.max()-allgr.min()+ew)
        lar=0.083*(2**(3.7*vsh)-1)
        set_feature(X,"Shale_indicator",rhob*nphi*lar)
    if not np.isfinite(X).all():raise RuntimeError("Non-finite engineered matrix.")
    return X

def descendant_audit(X0,X1,feature,tol=1e-10):
    allowed={p for n in DESCENDANTS[feature] for p in FEATURE_POSITIONS[n]}
    forbidden=sorted(set(range(len(LEGACY_FEATURE_COLUMNS)))-allowed)
    max_forbidden=float(np.max(np.abs(X1[:,forbidden]-X0[:,forbidden]))) if forbidden else 0.0
    return {"non_descendants_unchanged":bool(max_forbidden<=tol),"max_non_descendant_change":max_forbidden,
            "structural_zero_exact":bool(np.array_equal(X0,X1)) if feature in STRUCTURAL_ZERO else None}

In [12]:
def build_row_effects(bank,control,Pi_raw,Pi_cal,alpha,source_code,target_code):
    # Use identical numerical precision for control and intervention
    # probabilities before log-loss differences are computed.
    Pi_raw=np.asarray(Pi_raw,dtype=np.float64)
    Pi_cal=np.asarray(Pi_cal,dtype=np.float64)

    si=CLASS_TO_INDEX[source_code]; ti=CLASS_TO_INDEX[target_code]
    Pc_raw=control[RAW_PROB_COLS].to_numpy(float); Pc_cal=control[CAL_PROB_COLS].to_numpy(float)
    pcrs=Pc_raw[:,si]; pirs=Pi_raw[:,si]; pccs=Pc_cal[:,si]; pics=Pi_cal[:,si]
    pcrt=Pc_raw[:,ti]; pirt=Pi_raw[:,ti]; pcct=Pc_cal[:,ti]; pict=Pi_cal[:,ti]
    pred_c=CLASSES[np.argmax(Pc_raw,1)]
    pred_i=CLASSES[np.argmax(Pi_raw,1)]
    pred_i_cal=CLASSES[np.argmax(Pi_cal,1)]

    top_two=np.partition(Pi_raw,-2,axis=1)[:,-2:]
    raw_top2_margin=top_two[:,1]-top_two[:,0]

    calibrated_argmax_differs=pred_i!=pred_i_cal
    non_tie_difference=(
        calibrated_argmax_differs
        & (raw_top2_margin>ARGMAX_TIE_TOL)
    )

    if non_tie_difference.any():
        raise RuntimeError(
            "Calibration changed intervention argmax outside a numerical tie."
        )

    correct_c=pred_c==source_code
    correct_i=pred_i==source_code
    return pd.DataFrame({
      "ORIGINAL_ROW_ID":bank.ORIGINAL_ROW_ID.to_numpy(np.int64),"WELL":bank.WELL.to_numpy(),"DEPTH_MD":bank.DEPTH_MD.to_numpy(float),
      "outer_fold":bank.outer_fold.to_numpy(np.int64),"feature":bank.feature.to_numpy(),"source_code":source_code,"source_name":bank.source_name.to_numpy(),
      "target_code":target_code,"target_name":bank.target_name.to_numpy(),"alpha":float(alpha),"transport_space":bank.transport_space.to_numpy(),
      "original_value":bank.original_value.to_numpy(float),"intervened_value":bank[dose_col(alpha)].to_numpy(float),
      "target_quantile_value":bank.target_quantile_value.to_numpy(float),"source_percentile":bank.source_percentile.to_numpy(float),
      "normalized_full_transport_shift":bank.normalized_full_transport_shift.to_numpy(float),
      "control_predicted_code":pred_c,"intervened_predicted_code":pred_i,
      "calibrated_intervened_argmax_code":pred_i_cal,
      "calibrated_argmax_differs_from_raw":calibrated_argmax_differs,
      "raw_top2_probability_margin":raw_top2_margin,
      "calibration_argmax_difference_is_near_tie":(
          (~calibrated_argmax_differs)
          | (raw_top2_margin<=ARGMAX_TIE_TOL)
      ),
      "control_correct":correct_c,"intervened_correct":correct_i,
      "accuracy_change":correct_i.astype(np.int8)-correct_c.astype(np.int8),"any_flip":pred_i!=pred_c,
      "targeted_source_to_target_flip":(pred_c==source_code)&(pred_i==target_code),
      "target_acquired_from_non_target":(pred_c!=target_code)&(pred_i==target_code),
      "correct_to_incorrect":correct_c&(~correct_i),"incorrect_to_correct":(~correct_c)&correct_i,
      "raw_delta_source_probability":pirs-pcrs,"raw_delta_target_probability":pirt-pcrt,
      "raw_control_log_loss":-np.log(np.clip(pcrs,EPS,1)),"raw_intervened_log_loss":-np.log(np.clip(pirs,EPS,1)),
      "raw_delta_log_loss":-np.log(np.clip(pirs,EPS,1))+np.log(np.clip(pcrs,EPS,1)),
      "calibrated_delta_source_probability":pics-pccs,"calibrated_delta_target_probability":pict-pcct,
      "calibrated_control_log_loss":-np.log(np.clip(pccs,EPS,1)),"calibrated_intervened_log_loss":-np.log(np.clip(pics,EPS,1)),
      "calibrated_delta_log_loss":-np.log(np.clip(pics,EPS,1))+np.log(np.clip(pccs,EPS,1)),
    })

def per_well_summary(rows):
    return rows.groupby(["WELL","outer_fold","feature","source_code","source_name","target_code","target_name","alpha"],as_index=False).agg(
      rows=("ORIGINAL_ROW_ID","size"),baseline_accuracy=("control_correct","mean"),intervention_accuracy=("intervened_correct","mean"),
      accuracy_change=("accuracy_change","mean"),mean_raw_delta_log_loss=("raw_delta_log_loss","mean"),
      mean_calibrated_delta_log_loss=("calibrated_delta_log_loss","mean"),mean_raw_delta_source_probability=("raw_delta_source_probability","mean"),
      mean_calibrated_delta_source_probability=("calibrated_delta_source_probability","mean"),mean_raw_delta_target_probability=("raw_delta_target_probability","mean"),
      mean_calibrated_delta_target_probability=("calibrated_delta_target_probability","mean"),any_flip_rate=("any_flip","mean"),
      targeted_flip_rate=("targeted_source_to_target_flip","mean"),target_acquisition_rate=("target_acquired_from_non_target","mean"),
      correct_to_incorrect_rate=("correct_to_incorrect","mean"),incorrect_to_correct_rate=("incorrect_to_correct","mean"),
      mean_normalized_transport_shift=("normalized_full_transport_shift","mean"))

def dose_summary(rows,wells,condition_id,alpha):
    raw_mean,raw_lo,raw_hi=bootstrap_ci(wells.mean_raw_delta_log_loss,BOOTSTRAP_REPLICATES,stable_seed(condition_id,alpha,"raw"))
    cal_mean,cal_lo,cal_hi=bootstrap_ci(wells.mean_calibrated_delta_log_loss,BOOTSTRAP_REPLICATES,stable_seed(condition_id,alpha,"cal"))
    base_correct=int(rows.control_correct.sum()); base_wrong=int((~rows.control_correct).sum())
    c2i=float(rows.loc[rows.control_correct,"correct_to_incorrect"].mean()) if base_correct else np.nan
    i2c=float(rows.loc[~rows.control_correct,"incorrect_to_correct"].mean()) if base_wrong else np.nan
    return {
      "condition_id":condition_id,"outer_fold":int(rows.outer_fold.iloc[0]),"feature":rows.feature.iloc[0],"source_code":int(rows.source_code.iloc[0]),
      "source_name":rows.source_name.iloc[0],"target_code":int(rows.target_code.iloc[0]),"target_name":rows.target_name.iloc[0],"alpha":float(alpha),
      "rows":len(rows),"wells":rows.WELL.nunique(),"pooled_baseline_accuracy":float(rows.control_correct.mean()),
      "pooled_intervention_accuracy":float(rows.intervened_correct.mean()),"pooled_accuracy_change":float(rows.accuracy_change.mean()),
      "well_balanced_accuracy_change":float(wells.accuracy_change.mean()),"pooled_mean_raw_delta_log_loss":float(rows.raw_delta_log_loss.mean()),
      "well_balanced_mean_raw_delta_log_loss":raw_mean,"raw_CI_lower":raw_lo,"raw_CI_upper":raw_hi,"raw_wilcoxon_pvalue":safe_wilcoxon(wells.mean_raw_delta_log_loss),
      "pooled_mean_calibrated_delta_log_loss":float(rows.calibrated_delta_log_loss.mean()),"well_balanced_mean_calibrated_delta_log_loss":cal_mean,
      "calibrated_CI_lower":cal_lo,"calibrated_CI_upper":cal_hi,"calibrated_wilcoxon_pvalue":safe_wilcoxon(wells.mean_calibrated_delta_log_loss),
      "mean_raw_delta_source_probability":float(rows.raw_delta_source_probability.mean()),"mean_calibrated_delta_source_probability":float(rows.calibrated_delta_source_probability.mean()),
      "mean_raw_delta_target_probability":float(rows.raw_delta_target_probability.mean()),"mean_calibrated_delta_target_probability":float(rows.calibrated_delta_target_probability.mean()),
      "pooled_any_flip_rate":float(rows.any_flip.mean()),"well_balanced_any_flip_rate":float(wells.any_flip_rate.mean()),
      "pooled_targeted_flip_rate":float(rows.targeted_source_to_target_flip.mean()),"well_balanced_targeted_flip_rate":float(wells.targeted_flip_rate.mean()),
      "correct_to_incorrect_rate_given_control_correct":c2i,"incorrect_to_correct_rate_given_control_incorrect":i2c,
      "fraction_wells_positive_calibrated_delta_log_loss":float((wells.mean_calibrated_delta_log_loss>0).mean()),
      "mean_normalized_transport_shift":float(rows.normalized_full_transport_shift.mean()),"bootstrap_replicates":BOOTSTRAP_REPLICATES,
    }

In [13]:
replay_records=[]; summary_records=[]; well_frames=[]; transition_frames=[]; gate_records=[]; execution_records=[]
start_all=time.time()
RAW_PROB_COLS=[f"prob_{c}" for c in CLASSES]
CAL_PROB_COLS=[f"cal_prob_{c}" for c in CLASSES]

for fold in ACTIVE_FOLDS:
    print("\n"+"="*100); print("OUTER FOLD",fold); print("="*100)
    prefix=f"legacy_exact__outer_fold_{fold}"
    model_path=ARTIFACT_DIR/f"{prefix}__random_forest.joblib"
    scaler_path=ARTIFACT_DIR/f"{prefix}__scaler.joblib"
    imputer_path=ARTIFACT_DIR/f"{prefix}__imputers.joblib"
    raw_oof_path=OOF02B/f"{prefix}__oof.parquet"
    cal_oof_path=CAL_OOF03/f"{prefix}__nested_temperature_calibrated.parquet"
    for p in [model_path,scaler_path,imputer_path,raw_oof_path,cal_oof_path]:
        if not p.exists():raise FileNotFoundError(str(p))
    model=joblib.load(model_path); scaler=joblib.load(scaler_path); imputers=joblib.load(imputer_path)
    T=float(temperature_registry[str(fold)])
    raw_val=train.loc[train.GROUP_FOLD==fold].drop(columns=["GROUP_FOLD"]).copy().reset_index(drop=True)
    bundle=build_legacy_validation_matrix(raw_val,imputers)
    # build_legacy_validation_matrix returns X/y/metadata; obtain imputed frame separately for descendant context
    imputed=apply_train_imputers_to_validation(raw_val,imputers)
    Xbase=bundle["X"].to_numpy(float); metadata=bundle["metadata"]; y=bundle["y"]
    if model.n_features_in_!=26 or scaler.n_features_in_!=26:raise RuntimeError("Legacy feature count mismatch.")
    Praw=predict_aligned_in_batches(model,scaler.transform(Xbase),PREDICTION_BATCH_SIZE)
    Pcal=apply_temperature(Praw,T)
    stored_raw=pd.read_parquet(raw_oof_path); stored_cal=pd.read_parquet(cal_oof_path)
    Sr=stored_raw[RAW_PROB_COLS].to_numpy(float)
    Sc=stored_cal[[f"nested_calibrated_prob_{c}" for c in CLASSES]].to_numpy(float)
    replay={"outer_fold":fold,"rows":len(metadata),"wells":metadata.WELL.nunique(),"temperature":T,
      "well_order_match":bool(np.array_equal(metadata.WELL.astype(str),stored_raw.WELL.astype(str))),
      "depth_order_match":bool(np.allclose(metadata.DEPTH_MD,stored_raw.DEPTH_MD,rtol=0,atol=1e-10)),
      "raw_replay_match":bool(np.allclose(Praw,Sr,rtol=1e-6,atol=1e-6)),
      "calibrated_replay_match":bool(np.allclose(Pcal,Sc,rtol=1e-6,atol=1e-6)),
      "max_abs_raw_diff":float(np.max(np.abs(Praw-Sr))),"max_abs_cal_diff":float(np.max(np.abs(Pcal-Sc))),
      "frozen_model_scaler_imputer_temperature":True,"official_test_unused":True}
    replay["fold_replay_passed"]=bool(all(v for k,v in replay.items() if k not in ["outer_fold","rows","wells","temperature","max_abs_raw_diff","max_abs_cal_diff","fold_replay_passed"]))
    if not replay["fold_replay_passed"]:raise RuntimeError(f"Replay failed fold {fold}")
    replay_records.append(replay); print(json.dumps(replay,indent=2))
    ctx=build_context(imputed)
    id_to_pos=pd.Series(np.arange(len(metadata),dtype=np.int64),index=metadata.ORIGINAL_ROW_ID.to_numpy(np.int64))
    fold_conditions=selected[selected.outer_fold==fold]
    for ordinal,(_,cond) in enumerate(fold_conditions.iterrows(),1):
        t0=time.time(); cid=str(cond.condition_id); feature=str(cond.feature)
        cache_json=COND_DIR/f"{cid}.json"; cache_well=WELL_DIR/f"{cid}.parquet"; cache_trans=COND_DIR/f"{cid}__transitions.parquet"
        cache_row=ROW_DIR/f"{cid}__row_effects.parquet"
        completed=[cache_json,cache_well,cache_trans]+([cache_row] if SAVE_ROW_LEVEL_RESULTS else [])
        if (
            SKIP_COMPLETED_CONDITIONS
            and feature not in STRUCTURAL_ZERO
            and all(
                path.exists()
                for path in completed
            )
        ):
            cached=json.load(open(cache_json,encoding="utf-8"))
            if not cached["condition_gate_passed"]:raise RuntimeError(f"Cached gate failed {cid}")
            summary_records.extend(cached["dose_summaries"]); gate_records.append(cached["condition_gate"])
            well_frames.append(pd.read_parquet(cache_well)); transition_frames.append(pd.read_parquet(cache_trans))
            execution_records.append({"condition_id":cid,"outer_fold":fold,"status":"cache","seconds":0.0})
            print("[cache]",cid); continue
        print(f"[{ordinal}/{len(fold_conditions)}]",cid)
        bank_path=BANK_DIR/f"{cid}.parquet"; observed_hash=sha256_file(bank_path); expected_hash=str(cond.bank_sha256)
        if expected_hash!="nan" and observed_hash!=expected_hash:raise RuntimeError("Bank hash mismatch.")
        bank=pd.read_parquet(bank_path).sort_values("ORIGINAL_ROW_ID").reset_index(drop=True)
        needed={"ORIGINAL_ROW_ID","WELL","DEPTH_MD","outer_fold","feature","source_code","source_name","target_code","target_name",
                "transport_space","original_value","target_quantile_value","source_percentile","normalized_full_transport_shift"}|{dose_col(a) for a in DOSES}
        if needed-set(bank.columns):raise KeyError(f"Bank columns missing: {needed-set(bank.columns)}")
        ids=bank.ORIGINAL_ROW_ID.to_numpy(np.int64)
        if not np.isin(ids,id_to_pos.index).all():raise RuntimeError("Bank rows outside fold.")
        pos=id_to_pos.loc[ids].to_numpy(np.int64)
        source_code=int(bank.source_code.iloc[0]); target_code=int(bank.target_code.iloc[0])
        if not (metadata.iloc[pos][TARGET_COLUMN].to_numpy(np.int64)==source_code).all():raise RuntimeError("Source label mismatch.")
        original_imputed=pd.to_numeric(imputed.iloc[pos][feature],errors="raise").to_numpy(float)
        if not np.allclose(original_imputed,bank.original_value.to_numpy(float),rtol=1e-6,atol=1e-6):raise RuntimeError("Original value mismatch.")
        X0=Xbase[pos].copy()
        control=pd.DataFrame({"ORIGINAL_ROW_ID":ids})
        for j,c in enumerate(CLASSES):
            control[f"prob_{c}"]=Praw[pos,j]; control[f"cal_prob_{c}"]=Pcal[pos,j]
        matrices=[]; descendant_checks=[]
        for alpha in DOSES:
            Xcf=recompute_descendants(X0,feature,bank[dose_col(alpha)].to_numpy(float),pos,ctx)
            audit=descendant_audit(X0,Xcf,feature); audit["alpha"]=float(alpha); descendant_checks.append(audit); matrices.append(Xcf)
        Xstack=np.vstack(matrices)
        if feature in STRUCTURAL_ZERO:
            Pi_raw_all=np.tile(Praw[pos],(len(DOSES),1)); Pi_cal_all=np.tile(Pcal[pos],(len(DOSES),1))
        else:
            Pi_raw_all=predict_aligned_in_batches(model,scaler.transform(Xstack),PREDICTION_BATCH_SIZE)
            Pi_cal_all=apply_temperature(Pi_raw_all,T)
        n=len(bank); dose_summaries=[]; row_parts=[]; well_parts=[]; trans_parts=[]; prob_checks=[]
        for j,alpha in enumerate(DOSES):
            Pi_raw=Pi_raw_all[j*n:(j+1)*n]; Pi_cal=Pi_cal_all[j*n:(j+1)*n]
            pa=temperature_argmax_audit(Pi_raw,Pi_cal); pa["alpha"]=float(alpha); prob_checks.append(pa)
            rows=build_row_effects(bank,control,Pi_raw,Pi_cal,alpha,source_code,target_code)
            wells=per_well_summary(rows); dose_summaries.append(dose_summary(rows,wells,cid,alpha))
            trans=rows.groupby(["outer_fold","feature","source_code","source_name","target_code","target_name","alpha","control_predicted_code","intervened_predicted_code"],as_index=False).agg(rows=("ORIGINAL_ROW_ID","size"),wells=("WELL","nunique"))
            trans["control_predicted_name"]=trans.control_predicted_code.map(NAMES); trans["intervened_predicted_name"]=trans.intervened_predicted_code.map(NAMES)
            row_parts.append(rows); well_parts.append(wells); trans_parts.append(trans)
        rows_all=pd.concat(row_parts,ignore_index=True); wells_all=pd.concat(well_parts,ignore_index=True); trans_all=pd.concat(trans_parts,ignore_index=True)
        gate={"condition_id":cid,"outer_fold":fold,"feature":feature,"source_code":source_code,"target_code":target_code,
          "bank_hash_matches":observed_hash==expected_hash,"source_labels_match":True,"original_values_match":True,
          "independent_row_intervention":True,"single_raw_log_treated":True,"true_label_unchanged":True,
          "descendants_recomputed":True,"all_non_descendants_unchanged":bool(all(x["non_descendants_unchanged"] for x in descendant_checks)),
          "all_probability_checks":bool(all(x["probabilities_valid"] for x in prob_checks)),
          "calibration_argmax_mismatches_only_near_ties":bool(all(x["all_argmax_mismatches_are_near_ties"] for x in prob_checks)),
          "structural_zero_expected":feature in STRUCTURAL_ZERO,
          "structural_zero_exact":bool(
              np.array_equal(
                  Xstack,
                  np.tile(
                      X0,
                      (len(DOSES),1),
                  ),
              )
          ) if feature in STRUCTURAL_ZERO else None,
          "structural_zero_probability_exact":bool(
              np.array_equal(
                  Pi_raw_all,
                  np.tile(
                      Praw[pos],
                      (len(DOSES),1),
                  ),
              )
              and np.array_equal(
                  Pi_cal_all,
                  np.tile(
                      Pcal[pos],
                      (len(DOSES),1),
                  ),
              )
          ) if feature in STRUCTURAL_ZERO else None,
          "frozen_model_scaler_imputer_temperature":True,"legacy_refined_labels_unused":True,"official_test_unused":True}
        bools=[v for k,v in gate.items() if isinstance(v,bool) and k not in ["structural_zero_expected","condition_gate_passed"]]
        gate["condition_gate_passed"]=bool(all(bools))
        if not gate["condition_gate_passed"]:raise RuntimeError(f"Condition gate failed {cid}")
        wells_all.to_parquet(cache_well,index=False); trans_all.to_parquet(cache_trans,index=False)
        if SAVE_ROW_LEVEL_RESULTS:rows_all.to_parquet(cache_row,index=False)
        payload={"condition_id":cid,"dose_summaries":[json_safe(x) for x in dose_summaries],
                 "descendant_audits":[json_safe(x) for x in descendant_checks],"probability_audits":[json_safe(x) for x in prob_checks],
                 "condition_gate":json_safe(gate),"condition_gate_passed":True,"bank_sha256":observed_hash,
                 "row_level_saved":SAVE_ROW_LEVEL_RESULTS,"seconds":time.time()-t0}
        json.dump(payload,open(cache_json,"w",encoding="utf-8"),ensure_ascii=False,indent=2)
        summary_records.extend(dose_summaries); gate_records.append(gate); well_frames.append(wells_all); transition_frames.append(trans_all)
        execution_records.append({"condition_id":cid,"outer_fold":fold,"feature":feature,"rows":n,"wells":bank.WELL.nunique(),"status":"built","seconds":time.time()-t0})
        print(" PASS rows",n,"wells",bank.WELL.nunique(),"seconds",round(time.time()-t0,1))
        del bank,X0,control,matrices,Xstack,Pi_raw_all,Pi_cal_all,row_parts,well_parts,trans_parts,rows_all,wells_all,trans_all
        gc.collect()
    del model,scaler,imputers,raw_val,bundle,imputed,Xbase,metadata,Praw,Pcal,stored_raw,stored_cal,Sr,Sc,ctx,id_to_pos
    gc.collect()
print("Total seconds:",time.time()-start_all)


OUTER FOLD 1


[Parallel(n_jobs=10)]: Using backend ThreadingBackend with 10 concurrent workers.
[Parallel(n_jobs=10)]: Done  30 tasks      | elapsed:    0.1s
[Parallel(n_jobs=10)]: Done 180 tasks      | elapsed:    0.4s
[Parallel(n_jobs=10)]: Done 430 tasks      | elapsed:    0.8s
[Parallel(n_jobs=10)]: Done 780 tasks      | elapsed:    1.5s
[Parallel(n_jobs=10)]: Done 800 out of 800 | elapsed:    1.5s finished
[Parallel(n_jobs=10)]: Using backend ThreadingBackend with 10 concurrent workers.
[Parallel(n_jobs=10)]: Done  30 tasks      | elapsed:    0.0s
[Parallel(n_jobs=10)]: Done 180 tasks      | elapsed:    0.2s
[Parallel(n_jobs=10)]: Done 430 tasks      | elapsed:    0.6s
[Parallel(n_jobs=10)]: Done 780 tasks      | elapsed:    1.0s
[Parallel(n_jobs=10)]: Done 800 out of 800 | elapsed:    1.0s finished
[Parallel(n_jobs=10)]: Using backend ThreadingBackend with 10 concurrent workers.
[Parallel(n_jobs=10)]: Done  30 tasks      | elapsed:    0.0s
[Parallel(n_jobs=10)]: Done 180 tasks      | elapsed: 

{
  "outer_fold": 1,
  "rows": 232674,
  "wells": 20,
  "temperature": 0.7817853078382972,
  "well_order_match": true,
  "depth_order_match": true,
  "raw_replay_match": true,
  "calibrated_replay_match": true,
  "max_abs_raw_diff": 0.0,
  "max_abs_cal_diff": 2.384185791015625e-07,
  "frozen_model_scaler_imputer_temperature": true,
  "official_test_unused": true,
  "fold_replay_passed": true
}
[cache] fold_1__feature_CALI__source_74000_Dolomite__target_70000_Limestone
[cache] fold_1__feature_CALI__source_70000_Limestone__target_74000_Dolomite
[cache] fold_1__feature_CALI__source_70000_Limestone__target_65000_Shale
[cache] fold_1__feature_CALI__source_30000_Sandstone__target_65000_Shale
[cache] fold_1__feature_CALI__source_65030_Sandstone_Shale__target_65000_Shale
[cache] fold_1__feature_CALI__source_65000_Shale__target_70000_Limestone
[cache] fold_1__feature_CALI__source_65000_Shale__target_30000_Sandstone
[cache] fold_1__feature_CALI__source_65000_Shale__target_65030_Sandstone_Shale
[

[Parallel(n_jobs=10)]: Using backend ThreadingBackend with 10 concurrent workers.
[Parallel(n_jobs=10)]: Done  30 tasks      | elapsed:    0.1s
[Parallel(n_jobs=10)]: Done 180 tasks      | elapsed:    0.3s
[Parallel(n_jobs=10)]: Done 430 tasks      | elapsed:    0.8s
[Parallel(n_jobs=10)]: Done 780 tasks      | elapsed:    1.5s
[Parallel(n_jobs=10)]: Done 800 out of 800 | elapsed:    1.5s finished
[Parallel(n_jobs=10)]: Using backend ThreadingBackend with 10 concurrent workers.
[Parallel(n_jobs=10)]: Done  30 tasks      | elapsed:    0.1s
[Parallel(n_jobs=10)]: Done 180 tasks      | elapsed:    0.2s
[Parallel(n_jobs=10)]: Done 430 tasks      | elapsed:    0.5s
[Parallel(n_jobs=10)]: Done 780 tasks      | elapsed:    0.9s
[Parallel(n_jobs=10)]: Done 800 out of 800 | elapsed:    0.9s finished
[Parallel(n_jobs=10)]: Using backend ThreadingBackend with 10 concurrent workers.
[Parallel(n_jobs=10)]: Done  30 tasks      | elapsed:    0.0s
[Parallel(n_jobs=10)]: Done 180 tasks      | elapsed: 

{
  "outer_fold": 2,
  "rows": 236205,
  "wells": 20,
  "temperature": 0.8455709005107059,
  "well_order_match": true,
  "depth_order_match": true,
  "raw_replay_match": true,
  "calibrated_replay_match": true,
  "max_abs_raw_diff": 0.0,
  "max_abs_cal_diff": 2.384185791015625e-07,
  "frozen_model_scaler_imputer_temperature": true,
  "official_test_unused": true,
  "fold_replay_passed": true
}
[cache] fold_2__feature_CALI__source_74000_Dolomite__target_70000_Limestone
[cache] fold_2__feature_CALI__source_70000_Limestone__target_74000_Dolomite
[cache] fold_2__feature_CALI__source_70000_Limestone__target_65000_Shale
[cache] fold_2__feature_CALI__source_30000_Sandstone__target_65000_Shale
[cache] fold_2__feature_CALI__source_65030_Sandstone_Shale__target_65000_Shale
[cache] fold_2__feature_CALI__source_65000_Shale__target_70000_Limestone
[cache] fold_2__feature_CALI__source_65000_Shale__target_30000_Sandstone
[cache] fold_2__feature_CALI__source_65000_Shale__target_65030_Sandstone_Shale
[

...

{
  "outer_fold": 3,
  "rows": 230578,
  "wells": 21,
  "temperature": 0.7914888558618456,
  "well_order_match": true,
  "depth_order_match": true,
  "raw_replay_match": true,
  "calibrated_replay_match": true,
  "max_abs_raw_diff": 0.0,
  "max_abs_cal_diff": 2.384185791015625e-07,
  "frozen_model_scaler_imputer_temperature": true,
  "official_test_unused": true,
  "fold_replay_passed": true
}
[cache] fold_3__feature_CALI__source_74000_Dolomite__target_70000_Limestone
[cache] fold_3__feature_CALI__source_70000_Limestone__target_74000_Dolomite
[cache] fold_3__feature_CALI__source_70000_Limestone__target_65000_Shale
[cache] fold_3__feature_CALI__source_30000_Sandstone__target_65000_Shale
[cache] fold_3__feature_CALI__source_65030_Sandstone_Shale__target_65000_Shale
[cache] fold_3__feature_CALI__source_65000_Shale__target_70000_Limestone
[cache] fold_3__feature_CALI__source_65000_Shale__target_30000_Sandstone
[cache] fold_3__feature_CALI__source_65000_Shale__target_65030_Sandstone_Shale
[

[Parallel(n_jobs=10)]: Using backend ThreadingBackend with 10 concurrent workers.
[Parallel(n_jobs=10)]: Done  30 tasks      | elapsed:    0.1s
[Parallel(n_jobs=10)]: Done 180 tasks      | elapsed:    0.3s
[Parallel(n_jobs=10)]: Done 430 tasks      | elapsed:    0.8s
[Parallel(n_jobs=10)]: Done 780 tasks      | elapsed:    1.5s
[Parallel(n_jobs=10)]: Done 800 out of 800 | elapsed:    1.5s finished
[Parallel(n_jobs=10)]: Using backend ThreadingBackend with 10 concurrent workers.
[Parallel(n_jobs=10)]: Done  30 tasks      | elapsed:    0.1s
[Parallel(n_jobs=10)]: Done 180 tasks      | elapsed:    0.4s
[Parallel(n_jobs=10)]: Done 430 tasks      | elapsed:    1.0s
[Parallel(n_jobs=10)]: Done 780 tasks      | elapsed:    1.8s
[Parallel(n_jobs=10)]: Done 800 out of 800 | elapsed:    1.9s finished
[Parallel(n_jobs=10)]: Using backend ThreadingBackend with 10 concurrent workers.
[Parallel(n_jobs=10)]: Done  30 tasks      | elapsed:    0.0s
[Parallel(n_jobs=10)]: Done 180 tasks      | elapsed: 

{
  "outer_fold": 4,
  "rows": 237132,
  "wells": 17,
  "temperature": 0.8141043111357217,
  "well_order_match": true,
  "depth_order_match": true,
  "raw_replay_match": true,
  "calibrated_replay_match": true,
  "max_abs_raw_diff": 0.0,
  "max_abs_cal_diff": 2.384185791015625e-07,
  "frozen_model_scaler_imputer_temperature": true,
  "official_test_unused": true,
  "fold_replay_passed": true
}
[cache] fold_4__feature_CALI__source_74000_Dolomite__target_70000_Limestone
[cache] fold_4__feature_CALI__source_70000_Limestone__target_74000_Dolomite
[cache] fold_4__feature_CALI__source_70000_Limestone__target_65000_Shale
[cache] fold_4__feature_CALI__source_30000_Sandstone__target_65000_Shale
[cache] fold_4__feature_CALI__source_65030_Sandstone_Shale__target_65000_Shale
[cache] fold_4__feature_CALI__source_65000_Shale__target_70000_Limestone
[cache] fold_4__feature_CALI__source_65000_Shale__target_30000_Sandstone
[cache] fold_4__feature_CALI__source_65000_Shale__target_65030_Sandstone_Shale
[

[Parallel(n_jobs=10)]: Using backend ThreadingBackend with 10 concurrent workers.
[Parallel(n_jobs=10)]: Done  30 tasks      | elapsed:    0.0s
[Parallel(n_jobs=10)]: Done 180 tasks      | elapsed:    0.3s
[Parallel(n_jobs=10)]: Done 430 tasks      | elapsed:    0.7s
[Parallel(n_jobs=10)]: Done 780 tasks      | elapsed:    1.3s
[Parallel(n_jobs=10)]: Done 800 out of 800 | elapsed:    1.3s finished
[Parallel(n_jobs=10)]: Using backend ThreadingBackend with 10 concurrent workers.
[Parallel(n_jobs=10)]: Done  30 tasks      | elapsed:    0.0s
[Parallel(n_jobs=10)]: Done 180 tasks      | elapsed:    0.2s
[Parallel(n_jobs=10)]: Done 430 tasks      | elapsed:    0.5s
[Parallel(n_jobs=10)]: Done 780 tasks      | elapsed:    1.0s
[Parallel(n_jobs=10)]: Done 800 out of 800 | elapsed:    1.0s finished
[Parallel(n_jobs=10)]: Using backend ThreadingBackend with 10 concurrent workers.
[Parallel(n_jobs=10)]: Done  30 tasks      | elapsed:    0.0s
[Parallel(n_jobs=10)]: Done 180 tasks      | elapsed: 

{
  "outer_fold": 5,
  "rows": 233922,
  "wells": 20,
  "temperature": 0.8105943496262197,
  "well_order_match": true,
  "depth_order_match": true,
  "raw_replay_match": true,
  "calibrated_replay_match": true,
  "max_abs_raw_diff": 0.0,
  "max_abs_cal_diff": 2.384185791015625e-07,
  "frozen_model_scaler_imputer_temperature": true,
  "official_test_unused": true,
  "fold_replay_passed": true
}
[cache] fold_5__feature_CALI__source_74000_Dolomite__target_70000_Limestone
[cache] fold_5__feature_CALI__source_70000_Limestone__target_74000_Dolomite
[cache] fold_5__feature_CALI__source_70000_Limestone__target_65000_Shale
[cache] fold_5__feature_CALI__source_30000_Sandstone__target_65000_Shale
[cache] fold_5__feature_CALI__source_65030_Sandstone_Shale__target_65000_Shale
[cache] fold_5__feature_CALI__source_65000_Shale__target_70000_Limestone
[cache] fold_5__feature_CALI__source_65000_Shale__target_30000_Sandstone
[cache] fold_5__feature_CALI__source_65000_Shale__target_65030_Sandstone_Shale
[

In [14]:
replay_df=pd.DataFrame(replay_records)
summary=pd.DataFrame(summary_records)
wells=pd.concat(well_frames,ignore_index=True)
transitions=pd.concat(transition_frames,ignore_index=True)
condition_gates=pd.DataFrame(gate_records)
execution=pd.DataFrame(execution_records)

for alpha in DOSES:
    m=np.isclose(summary.alpha,alpha)
    summary.loc[m,"raw_qvalue"]=bh(summary.loc[m,"raw_wilcoxon_pvalue"])
    summary.loc[m,"calibrated_qvalue"]=bh(summary.loc[m,"calibrated_wilcoxon_pvalue"])
summary["raw_FDR_significant"]=summary.raw_qvalue<.05
summary["calibrated_FDR_significant"]=summary.calibrated_qvalue<.05
summary["calibrated_CI_width"]=summary.calibrated_CI_upper-summary.calibrated_CI_lower

dose_rows=[]
for cid,g in summary.groupby("condition_id"):
    g=g.sort_values("alpha"); x=g.alpha.to_numpy(float); yc=g.well_balanced_mean_calibrated_delta_log_loss.to_numpy(float); yr=g.well_balanced_mean_raw_delta_log_loss.to_numpy(float)
    sc=spearmanr(x,yc); sr=spearmanr(x,yr)
    dose_rows.append({"condition_id":cid,"outer_fold":int(g.outer_fold.iloc[0]),"feature":g.feature.iloc[0],"source_code":int(g.source_code.iloc[0]),"source_name":g.source_name.iloc[0],
      "target_code":int(g.target_code.iloc[0]),"target_name":g.target_name.iloc[0],"calibrated_spearman":float(sc.statistic),"calibrated_spearman_pvalue":float(sc.pvalue),
      "raw_spearman":float(sr.statistic),"raw_spearman_pvalue":float(sr.pvalue),"calibrated_non_decreasing":bool(np.all(np.diff(yc)>=-1e-12)),
      "raw_non_decreasing":bool(np.all(np.diff(yr)>=-1e-12)),"calibrated_alpha1_minus_alpha025":float(yc[-1]-yc[0]),"raw_alpha1_minus_alpha025":float(yr[-1]-yr[0])})
dose_response=pd.DataFrame(dose_rows)

lookup={(int(r.outer_fold),r.feature,int(r.source_code),int(r.target_code),float(r.alpha)):r for _,r in summary.iterrows()}
asym=[]
for _,r in summary.iterrows():
    rev=lookup.get((int(r.outer_fold),r.feature,int(r.target_code),int(r.source_code),float(r.alpha)))
    if rev is None:continue
    z=r.to_dict(); z.update({"reverse_condition_id":rev.condition_id,"reverse_source_code":int(rev.source_code),"reverse_source_name":rev.source_name,
      "reverse_target_code":int(rev.target_code),"reverse_target_name":rev.target_name,
      "reverse_cal_effect":float(rev.well_balanced_mean_calibrated_delta_log_loss),"reverse_raw_effect":float(rev.well_balanced_mean_raw_delta_log_loss),
      "reverse_wells":int(rev.wells),"reverse_rows":int(rev.rows),
      "calibrated_forward_minus_reverse":float(r.well_balanced_mean_calibrated_delta_log_loss-rev.well_balanced_mean_calibrated_delta_log_loss),
      "raw_forward_minus_reverse":float(r.well_balanced_mean_raw_delta_log_loss-rev.well_balanced_mean_raw_delta_log_loss)})
    asym.append(z)
asymmetry=pd.DataFrame(asym)

ranking=summary.groupby(["feature","alpha"],as_index=False).agg(
 conditions=("condition_id","nunique"),folds=("outer_fold","nunique"),
 median_signed_cal_effect=("well_balanced_mean_calibrated_delta_log_loss","median"),
 mean_signed_cal_effect=("well_balanced_mean_calibrated_delta_log_loss","mean"),
 median_absolute_cal_effect=("well_balanced_mean_calibrated_delta_log_loss",lambda x:float(np.median(np.abs(x)))),
 max_harmful_cal_effect=("well_balanced_mean_calibrated_delta_log_loss","max"),
 median_target_probability_change=("mean_calibrated_delta_target_probability","median"),
 median_targeted_flip_rate=("well_balanced_targeted_flip_rate","median"),
 FDR_significant_fraction=("calibrated_FDR_significant","mean"))

structural=summary[
    summary.feature.astype(str).isin(
        STRUCTURAL_ZERO
    )
].copy()

expected_doses=(
    expected_conditions
    * len(DOSES)
)

STRUCTURAL_ZERO_EFFECT_TOLERANCE=1e-12

selected_structural_conditions=selected.loc[
    selected.feature.astype(str).isin(
        STRUCTURAL_ZERO
    )
].copy()

expected_structural_condition_ids=set(
    selected_structural_conditions.condition_id.astype(str)
)

observed_structural_condition_ids=set(
    structural.condition_id.astype(str)
)

expected_structural_summary_rows=(
    len(expected_structural_condition_ids)
    * len(DOSES)
)

structural_condition_gates=condition_gates.loc[
    condition_gates.feature.astype(str).isin(
        STRUCTURAL_ZERO
    )
].copy()

structural_numeric_columns=[
    "well_balanced_mean_raw_delta_log_loss",
    "well_balanced_mean_calibrated_delta_log_loss",
    "pooled_mean_raw_delta_log_loss",
    "pooled_mean_calibrated_delta_log_loss",
    "pooled_accuracy_change",
    "well_balanced_accuracy_change",
    "pooled_any_flip_rate",
    "well_balanced_any_flip_rate",
    "pooled_targeted_flip_rate",
    "well_balanced_targeted_flip_rate",
]

structural_values_finite=bool(
    len(structural)>0
    and np.isfinite(
        structural[
            structural_numeric_columns
        ].to_numpy(float)
    ).all()
)

structural_max_abs_effect={
    column:(
        float(
            structural[column]
            .abs()
            .max()
        )
        if len(structural)>0
        else None
    )
    for column in structural_numeric_columns
}

matrix_exact=bool(
    len(structural_condition_gates)
    == len(expected_structural_condition_ids)
    and structural_condition_gates[
        "structural_zero_exact"
    ].fillna(False).astype(bool).all()
)

probability_exact=bool(
    len(structural_condition_gates)
    == len(expected_structural_condition_ids)
    and structural_condition_gates[
        "structural_zero_probability_exact"
    ].fillna(False).astype(bool).all()
)

structural_zero_diagnostics={
    "expected_features":sorted(
        set(
            selected_structural_conditions.feature.astype(str)
        )
    ),
    "observed_features":sorted(
        set(
            structural.feature.astype(str)
        )
    ),
    "expected_condition_ids":sorted(
        expected_structural_condition_ids
    ),
    "observed_condition_ids":sorted(
        observed_structural_condition_ids
    ),
    "condition_ids_match_registry":bool(
        observed_structural_condition_ids
        == expected_structural_condition_ids
    ),
    "expected_condition_count":int(
        len(expected_structural_condition_ids)
    ),
    "observed_condition_count":int(
        structural.condition_id.nunique()
    ),
    "expected_summary_rows":int(
        expected_structural_summary_rows
    ),
    "observed_summary_rows":int(
        len(structural)
    ),
    "summary_row_count_correct":bool(
        len(structural)
        == expected_structural_summary_rows
    ),
    "all_checked_values_finite":structural_values_finite,
    "effect_tolerance":float(
        STRUCTURAL_ZERO_EFFECT_TOLERANCE
    ),
    "maximum_absolute_values":structural_max_abs_effect,
    "engineered_matrix_exact":matrix_exact,
    "raw_and_calibrated_probabilities_exact":probability_exact,
    "exact_representation_and_probabilities_required":True,
}

structural_zero_diagnostics[
    "all_structural_effects_within_tolerance"
]=bool(
    structural_values_finite
    and all(
        value is not None
        and value<=STRUCTURAL_ZERO_EFFECT_TOLERANCE
        for value in structural_max_abs_effect.values()
    )
)

structural_zero_diagnostics[
    "structural_zero_controls_passed"
]=bool(
    len(expected_structural_condition_ids)>0
    and structural_zero_diagnostics[
        "condition_ids_match_registry"
    ]
    and structural_zero_diagnostics[
        "summary_row_count_correct"
    ]
    and structural_zero_diagnostics[
        "all_checked_values_finite"
    ]
    and structural_zero_diagnostics[
        "all_structural_effects_within_tolerance"
    ]
    and structural_zero_diagnostics[
        "engineered_matrix_exact"
    ]
    and structural_zero_diagnostics[
        "raw_and_calibrated_probabilities_exact"
    ]
)

stage_gate={"official_test_unused":True,"02B_legacy_model_used":True,"03_nested_temperatures_used":True,"04C_bank_used":True,"06A_registry_used":True,
 "all_fold_replays_passed":bool(replay_df.fold_replay_passed.all()),"expected_condition_count":expected_conditions,
 "observed_condition_count":int(condition_gates.condition_id.nunique()),"condition_count_correct":bool(condition_gates.condition_id.nunique()==expected_conditions),
 "all_condition_gates_passed":bool(condition_gates.condition_gate_passed.all()),"expected_condition_dose_count":expected_doses,
 "observed_condition_dose_count":len(summary),"condition_dose_count_correct":bool(len(summary)==expected_doses),
 "independent_row_interventions":True,"single_raw_log_at_a_time":True,"descendants_recomputed":True,
 "non_descendants_unchanged":bool(condition_gates.all_non_descendants_unchanged.all()),
 "PEF_SP_structural_zero_controls_passed":bool(structural_zero_diagnostics["structural_zero_controls_passed"]),
 "frozen_model_scaler_imputer_temperature":True,"legacy_refined_labels_unused":True,
 "well_balanced_primary_estimand":True,"positive_delta_log_loss_means_harm":True,"raw_and_calibrated_effects_saved":True,"run_mode":RUN_MODE}

boolean_types=(bool,np.bool_)
false_stage_gate_keys=[
    key
    for key,value in stage_gate.items()
    if isinstance(value,boolean_types)
    and not bool(value)
]

stage_gate[
    "failed_boolean_gate_keys"
]=false_stage_gate_keys

stage_gate[
    "structural_zero_diagnostics"
]=structural_zero_diagnostics

stage_gate[
    "overall_legacy_RF_transport_intervention_stage_passed"
]=bool(
    len(false_stage_gate_keys)==0
)

print(json.dumps(stage_gate,indent=2))

# Save diagnostics before a possible exception.
structural.to_csv(
    OUT/"11_structural_zero_controls.csv",
    index=False,
)

json.dump(
    json_safe(stage_gate),
    open(
        OUT/"12_stage_gate.json",
        "w",
        encoding="utf-8",
    ),
    ensure_ascii=False,
    indent=2,
)

if not stage_gate[
    "overall_legacy_RF_transport_intervention_stage_passed"
]:
    raise RuntimeError(
        "06B stage gate failed. False checks: "
        + ", ".join(false_stage_gate_keys)
        + ". See 12_stage_gate.json and "
        "11_structural_zero_controls.csv."
    )

replay_df.to_csv(OUT/"00_frozen_legacy_control_replay_audit.csv",index=False)
selected.to_csv(OUT/"01_selected_intervention_conditions.csv",index=False)
descendant_map.to_csv(OUT/"02_model_descendant_map.csv",index=False)
execution.to_csv(OUT/"03_condition_execution_manifest.csv",index=False)
condition_gates.to_csv(OUT/"04_condition_gates.csv",index=False)
summary.to_csv(OUT/"05_condition_dose_effect_summary.csv",index=False)
wells.to_csv(OUT/"06_all_per_well_effects.csv",index=False); wells.to_parquet(OUT/"06_all_per_well_effects.parquet",index=False)
dose_response.to_csv(OUT/"07_dose_response_summary.csv",index=False)
asymmetry.to_csv(OUT/"08_direction_asymmetry_summary.csv",index=False)
ranking.to_csv(OUT/"09_feature_effect_ranking.csv",index=False)
transitions.to_csv(OUT/"10_prediction_transition_summary.csv",index=False); transitions.to_parquet(OUT/"10_prediction_transition_summary.parquet",index=False)
structural.to_csv(OUT/"11_structural_zero_controls.csv",index=False)
json.dump(stage_gate,open(OUT/"12_stage_gate.json","w",encoding="utf-8"),ensure_ascii=False,indent=2)
config={"stage":"06B_legacy_RF_class_conditional_transport_interventions","run_mode":RUN_MODE,"official_test_used":False,
 "outer_folds":ACTIVE_FOLDS,"doses":DOSES,"selected_condition_count":expected_conditions,
 "legacy_feature_columns":LEGACY_FEATURE_COLUMNS,"feature_column_count":26,"unique_feature_count":25,"duplicate_feature_names":["X_LOC"],
 "intervention_unit":"independent held-out validation row","descendant_policy":"recompute only model descendants of the treated raw log for the treated row",
 "GR_flatness_policy":"original previous four rows in same WELL plus counterfactual current GR",
 "DTC_grad_policy":"counterfactual current DTC minus original previous DTC",
 "VSH_policy":"row-specific validation-fold extrema under a single-row GR intervention",
 "primary_outcome":"delta log loss = intervention loss - control loss","primary_estimand":"mean within WELL then equal mean across WELLs",
 "legacy_refined_labels_used":False,"model_refitted":False,"scaler_refitted":False,"imputer_refitted":False,"temperature_refitted":False,
 "row_level_saved":SAVE_ROW_LEVEL_RESULTS,"next_stage":"06C_raw_vs_legacy_representation_amplification"}
json.dump(config,open(OUT/"13_run_configuration.json","w",encoding="utf-8"),ensure_ascii=False,indent=2)
print("Saved:",OUT)
display(ranking[np.isclose(ranking.alpha,1.0)].sort_values("median_absolute_cal_effect",ascending=False))

{
  "official_test_unused": true,
  "02B_legacy_model_used": true,
  "03_nested_temperatures_used": true,
  "04C_bank_used": true,
  "06A_registry_used": true,
  "all_fold_replays_passed": true,
  "expected_condition_count": 319,
  "observed_condition_count": 319,
  "condition_count_correct": true,
  "all_condition_gates_passed": true,
  "expected_condition_dose_count": 1276,
  "observed_condition_dose_count": 1276,
  "condition_dose_count_correct": true,
  "independent_row_interventions": true,
  "single_raw_log_at_a_time": true,
  "descendants_recomputed": true,
  "non_descendants_unchanged": true,
  "PEF_SP_structural_zero_controls_passed": true,
  "frozen_model_scaler_imputer_temperature": true,
  "legacy_refined_labels_unused": true,
  "well_balanced_primary_estimand": true,
  "positive_delta_log_loss_means_harm": true,
  "raw_and_calibrated_effects_saved": true,
  "run_mode": "full",
  "failed_boolean_gate_keys": [],
  "structural_zero_diagnostics": {
    "expected_features": [
 

,feature,alpha,conditions,folds,median_signed_cal_effect,mean_signed_cal_effect,median_absolute_cal_effect,max_harmful_cal_effect,median_target_probability_change,median_targeted_flip_rate,FDR_significant_fraction
11,GR,1.0,40,5,0.382888,0.396317,0.639787,1.108029,0.082763,0.064858,0.825
15,NPHI,1.0,40,5,0.174441,0.168525,0.208089,0.517254,0.044544,0.023462,0.800
7,DTC,1.0,40,5,0.163385,0.047354,0.192334,0.421861,0.043702,0.023394,0.525
27,RHOB,1.0,40,5,0.043597,0.000050,0.059046,0.467757,0.004595,0.002751,0.625
3,CALI,1.0,40,5,0.023293,0.117922,0.036759,2.940436,0.004239,0.000947,0.575
23,RDEP,1.0,40,5,0.027922,0.005042,0.029856,0.676479,0.005528,0.003326,0.575
19,PEF,1.0,39,5,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000
31,SP,1.0,40,5,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000
